In [2]:
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential, clone_model
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape, Lambda
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
import numpy as np
import matplotlib.pyplot as plt

In [3]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

# Expand dims for channel
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

NUM_CLASSES = 10
y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

# Simple CNN classifier
def build_classifier(input_shape=(28,28,1), num_classes=10):
    model = Sequential([
        Conv2D(32, 3, activation="relu", input_shape=input_shape),
        MaxPooling2D(),
        Conv2D(64, 3, activation="relu"),
        MaxPooling2D(),
        Flatten(),
        Dense(128, activation="relu"),
        Dense(num_classes, activation="softmax")
    ])
    return model

classifier = build_classifier()
classifier.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
classifier.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=3, batch_size=128)

# Evaluate baseline
loss, acc = classifier.evaluate(x_test, y_test_cat)
print("Baseline test accuracy:", acc)

c:\Anaconda\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 25s 50ms/step - accuracy: 0.9378 - loss: 0.2082 - val_accuracy: 0.9819 - val_loss: 0.0637
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 23s 48ms/step - accuracy: 0.9825 - loss: 0.0552 - val_accuracy: 0.9851 - val_loss: 0.0436
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 19s 41ms/step - accuracy: 0.9883 - loss: 0.0379 - val_accuracy: 0.9882 - val_loss: 0.0363
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.9882 - loss: 0.0363
Baseline test accuracy: 0.9882000088691711


In [25]:
idx = 0
orig_image = x_test[idx:idx+1]  # Shape (1,28,28,1)
orig_label = y_test_cat[idx:idx+1]

# Heavy random noise
noise = np.random.normal(0, 0.9, size=orig_image.shape)
noisy_image = np.clip(orig_image + noise, 0.0, 1.0)

# Evaluate noisy image
pred = classifier.predict(noisy_image)
print("Predicted class (noisy):", np.argmax(pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 98ms/step
Predicted class (noisy): 8


In [26]:
# Freeze classifier
classifier.trainable = False

input_shape = orig_image.shape[1:]  # (28,28,1)
num_pixels = np.prod(input_shape)

# Input: single constant 1
noise_input = Input(shape=(1,), name="noise_input")

# Adversarial noise layer (fully connected, no bias)
adv_noise_layer = Dense(num_pixels, use_bias=False, kernel_regularizer=tf.keras.regularizers.l1(1e-3))(noise_input)

# Reshape to image
adv_noise_reshaped = Reshape(input_shape)(adv_noise_layer)

# Add to original image
orig_image_input = Input(shape=input_shape)
adv_image = Lambda(lambda x: tf.clip_by_value(x[0]+x[1], 0.0, 1.0))([orig_image_input, adv_noise_reshaped])

# Feed through classifier
output = classifier(adv_image)

# Define the model
adv_model = Model(inputs=[orig_image_input, noise_input], outputs=output)

In [27]:
#Custom loss fn
def non_targeted_loss(y_true, y_pred):
    return -tf.keras.losses.categorical_crossentropy(y_true, y_pred)

In [28]:
adv_model.compile(optimizer=Adam(1e-2), loss=non_targeted_loss)

# Constant input for noise layer
noise_const = np.ones((1,1))

# Train to misclassify the selected image
adv_model.fit([orig_image, noise_const], orig_label, epochs=50, verbose=1)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 610ms/step - loss: 0.0335
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 0.0267
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - loss: 0.0207
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - loss: 0.0155
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 0.0116
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.0088
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0069
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - loss: 0.0060
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 57ms/step - loss: 0.0060
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.0068
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.0071
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 0.0070
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - loss: 0.0066
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 54ms/step - loss: 0.0060
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0053
Epoch 16/50
1/1 ━━━━━━━━━━━━━━━━

In [29]:
adv_image_final = adv_model.predict([orig_image, noise_const])
pred_class = np.argmax(adv_image_final)
print("Original class:", np.argmax(orig_label))
print("Adversarial predicted class:", pred_class)

# Visualize
plt.figure(figsize=(5,5))
plt.subplot(1,2,1)
plt.title("Original")
plt.imshow(orig_image[0,:,:,0], cmap="gray")
plt.axis("off")

plt.subplot(1,2,2)
plt.title("Adversarial")
adv_noise = adv_model.get_layer(index=1).get_weights()[0].reshape(28,28,1)
adv_image_plot = np.clip(orig_image[0] + adv_noise, 0.0, 1.0)
plt.imshow(adv_image_plot[:,:,0], cmap='gray')
plt.axis("off")
plt.show()

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 105ms/step
Original class: 7
Adversarial predicted class: 7


C:\Users\suraj\AppData\Local\Temp\ipykernel_736\1774245527.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Steps
- Select an image and apply some heavy random noise to the image and evaluate the noisy image on the classification model as well.
- create a new model, which is a copy of the trained classifier, but with additional preprocessing layers, as depicted in the network diagram. The adversarial_noise layer is a Dense layer that is fully connected to a placeholder input with a single constant value of 1. 
The bias for this layer is turned off, so each neuron in the adversarial_noise layer outputs the same value as the weight connecting it to the unity input.
- As the network trains, each weight corresponds with the adversarial noise applied to a unique pixel in the image. Apply a regularization (e.g l1, l2) to keep the pixel perturbations small and/or innumerous.
- Re-shape the outputs to match the image shape, and add to the input image. Use a custom activation function is used to clip the values to be between 0.0 and 1.0. This is to prevent the model from applying noise that produces pixel values outside the scale of our input image. Finally, feed the augmented image through the classifier to obtain an output. The classifier weights are frozen so that only the adversarial_noise layer weights are trained.
- To achieve a non-targeted misclassification, use a custom loss function which is simply the negative of categorical_crossentropy. Then train the model using the original image as input and the true class as output. The custom loss function will minimize the accuracy, producing a non-targeted misclassification.
- For a targeted misclassification, use the categorical_crossentropy loss function and supply the model with our desired target as output during training.

In [ ]:
targeted_loss = tf.keras.losses.CategoricalCrossentropy()
adv_model.compile(optimizer=Adam(1e-2), loss=targeted_loss)
desired_target = np.zeros((1, NUM_CLASSES))
desired_target[0, (np.argmax(orig_label)+1) % NUM_CLASSES] = 1  # Target the next class
adv_model.fit([orig_image, noise_const], desired_target, epochs=50, verbose=1)  

adv_image_final = adv_model.predict([orig_image, noise_const])
pred_class = np.argmax(adv_image_final)
print("Original class:", np.argmax(orig_label))
print("Adversarial predicted class:", pred_class)

Epoch 1/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 606ms/step - loss: 13.6514
Epoch 2/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 13.1748
Epoch 3/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 12.5039
Epoch 4/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 11.7982
Epoch 5/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 11.0645
Epoch 6/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 65ms/step - loss: 10.3325
Epoch 7/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - loss: 9.6070
Epoch 8/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - loss: 8.8759
Epoch 9/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 8.1007
Epoch 10/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step - loss: 7.3289
Epoch 11/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 101ms/step - loss: 6.5700
Epoch 12/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - loss: 5.8207
Epoch 13/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - loss: 5.1080
Epoch 14/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - loss: 4.4360
Epoch 15/50
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - loss: 3.7726
Epoch 16/50
1/1 ━━━━━━━━━━

# Pixelwise nosie 


In [15]:
import tensorflow as tf
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Dense, Flatten, Reshape, Lambda, Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import categorical_crossentropy
import numpy as np
import matplotlib.pyplot as plt
import os

# Output directory for adversarial images
OUTPUT_DIR = "adv_images"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [16]:
# Load MNIST
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
x_train = np.expand_dims(x_train, axis=-1)
x_test = np.expand_dims(x_test, axis=-1)

NUM_CLASSES = 10
y_train_cat = tf.keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test_cat = tf.keras.utils.to_categorical(y_test, NUM_CLASSES)

# Simple CNN classifier
def build_classifier(input_shape=(28,28,1), num_classes=10):
    model = Sequential([
        Conv2D(32, 3, activation="relu", input_shape=input_shape),
        MaxPooling2D(),
        Conv2D(64, 3, activation="relu"),
        MaxPooling2D(),
        Flatten(),
        Dense(128, activation="relu"),
        Dense(num_classes, activation="softmax")
    ])
    return model

classifier = build_classifier()
classifier.compile(optimizer=Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
classifier.fit(x_train, y_train_cat, validation_data=(x_test, y_test_cat), epochs=3, batch_size=128)

classifier.trainable = False  # Freeze classifier for adversarial training

Epoch 1/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 22s 44ms/step - accuracy: 0.9396 - loss: 0.2106 - val_accuracy: 0.9800 - val_loss: 0.0646
Epoch 2/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 27s 56ms/step - accuracy: 0.9823 - loss: 0.0575 - val_accuracy: 0.9831 - val_loss: 0.0547
Epoch 3/3
469/469 ━━━━━━━━━━━━━━━━━━━━ 28s 58ms/step - accuracy: 0.9869 - loss: 0.0421 - val_accuracy: 0.9875 - val_loss: 0.0372


In [20]:
IMG_SHAPE = x_train.shape[1:]  # (28,28,1)
NUM_PIXELS = np.prod(IMG_SHAPE)

# Input: original image and a constant for noise layer
orig_image_input = Input(shape=IMG_SHAPE, name="orig_image")
noise_const_input = Input(shape=(1,), name="noise_const")

# Dense layer: each weight is pixel noise, no bias
adv_noise = Dense(NUM_PIXELS,
                  use_bias=False,
                  kernel_regularizer=tf.keras.regularizers.l1(1e-3))(noise_const_input)

# Reshape to image shape
adv_noise_reshaped = Reshape(IMG_SHAPE)(adv_noise)

# Clip the added noise to [0,1]
adv_image = Lambda(lambda x: tf.clip_by_value(x[0]+x[1], 0.0, 1.0))([orig_image_input, adv_noise_reshaped])

# Feed through frozen classifier
adv_output = classifier(adv_image)

# Adversarial model
adv_model = Model(inputs=[orig_image_input, noise_const_input], outputs=adv_output)

In [21]:
# Non-targeted misclassification (minimize classifier accuracy)
def non_targeted_loss(y_true, y_pred):
    return -categorical_crossentropy(y_true, y_pred)

# For targeted attacks, just use normal categorical_crossentropy
targeted_loss = categorical_crossentropy

In [22]:
class SaveAdvImage(tf.keras.callbacks.Callback):
    def __init__(self, orig_image, noise_const, output_dir=OUTPUT_DIR, prefix="adv_epoch"):
        super().__init__()
        self.orig_image = orig_image
        self.noise_const = noise_const
        self.output_dir = output_dir
        self.prefix = prefix
        os.makedirs(self.output_dir, exist_ok=True)
        
    def on_epoch_end(self, epoch, logs=None):
        adv_img = self.model.predict([self.orig_image, self.noise_const])
        # Compute adversarial image
        adv_noise_layer = self.model.get_layer(index=1).get_weights()[0].reshape(IMG_SHAPE)
        adv_image_plot = np.clip(self.orig_image[0] + adv_noise_layer, 0.0, 1.0)
        
        plt.figure(figsize=(4,4))
        plt.title(f"Epoch {epoch+1}")
        plt.imshow(adv_image_plot[:,:,0], cmap="gray")
        plt.axis("off")
        filename = os.path.join(self.output_dir, f"{self.prefix}_{epoch+1:03d}.png")
        plt.savefig(filename)
        plt.close()
        print(f"Saved adversarial image to {filename}")

In [23]:
# Choose an example image
idx = 0
orig_img = x_test[idx:idx+1]
orig_lbl = y_test_cat[idx:idx+1]
noise_const = np.ones((1,1))

# Compile adversarial model
adv_model.compile(optimizer=Adam(1e-2), loss=non_targeted_loss)

# Callback for saving images each epoch
save_cb = SaveAdvImage(orig_image=orig_img, noise_const=noise_const)

# Train
adv_model.fit([orig_img, noise_const], orig_lbl, epochs=20, callbacks=[save_cb], verbose=1)

Epoch 1/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/stepp - loss: 0.033
Saved adversarial image to adv_images\adv_epoch_001.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 860ms/step - loss: 0.0335
Epoch 2/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.026
Saved adversarial image to adv_images\adv_epoch_002.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 250ms/step - loss: 0.0265
Epoch 3/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - loss: 0.020
Saved adversarial image to adv_images\adv_epoch_003.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 261ms/step - loss: 0.0205
Epoch 4/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - loss: 0.015
Saved adversarial image to adv_images\adv_epoch_004.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 287ms/step - loss: 0.0156
Epoch 5/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 97ms/step - loss: 0.011
Saved adversarial image to adv_images\adv_epoch_005.png
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - loss: 0.0117
Epoch 6/20
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - loss: 0.008
Saved adversarial image to adv_images\adv_epoch_006.png
1/1 ━━━━━━━━━━━